In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 

In [ ]:
#!/usr/bin/env python3
"""
Evaluate the base, un-finetuned Qwen3.5 model on a test set to extract:
- Accuracy
- Macro F1
- Confusion Matrix
"""

import os
# Fix for the Unsloth Qwen3.5 Compiler Bug
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import json
import re
import argparse
import torch
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SYSTEM_PROMPT = (
    "You convert a shopping request into a structured product query. "
    "Respond with only a JSON object containing the field main_category.\n"
    "The main_category MUST be exactly one of the following allowed categories:\n"
    "- Arts, Crafts & Party Supplies\n"
    "- Automotive\n"
    "- Baby Products\n"
    "- Beauty & Personal Care\n"
    "- Electronics & Computers\n"
    "- Fashion, Shoes & Luggage\n"
    "- Health & Household\n"
    "- Home & Kitchen\n"
    "- Industrial & Scientific\n"
    "- Pet Supplies\n"
    "- Smart Home\n"
    "- Sports & Outdoors\n"
    "- Tools & Home Improvement\n"
    "- Toys & Games\n"
    "- Video Games"
)

def extract_category(text):
    """
    Attempts to clean up and extract the 'main_category' string from the model output.
    Uses JSON parsing first, and falls back to Regex if the JSON is malformed.
    """
    text = text.strip()
    # 1. Try direct JSON parsing
    try:
        data = json.loads(text)
        if "main_category" in data:
            return str(data["main_category"]).strip()
    except Exception:
        pass

    # 2. Fallback: Regex extraction if model wrapped it in markdown block or wrote messy JSON
    try:
        # Look for "main_category": "VALUE"
        match = re.search(r'"main_category"\s*:\s*"([^"]+)"', text)
        if match:
            return match.group(1).strip()
    except Exception:
        pass

    return "UNKNOWN"

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model-name", default="unsloth/Qwen3.5-0.8B")
    ap.add_argument("--test-file", default="test.jsonl")
    ap.add_argument("--max-seq-length", type=int, default=256)
    args, _ = ap.parse_known_args()

    # ---- 1. Load Base Model Only ----
    print(f"Loading base model: {args.model_name}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=args.model_name,
        max_seq_length=args.max_seq_length,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    # ---- 2. Load Dataset ----
    print(f"Loading test data from: {args.test_file}")
    ds = load_dataset("json", data_files={"test": args.test_file})["test"]

    y_true = []
    y_pred = []

# ---- 3. Inference Loop ----
    print("\nRunning evaluation on test data...")
    for item in tqdm(ds):
        query = item["query"]

        # Get true category
        true_cat = item.get("main_category", None)
        if not true_cat and "target" in item:
            true_cat = extract_category(item["target"])

        if not true_cat:
            continue

        # Format input prompt exactly like the training template
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ]

        # 1. Get raw prompt string
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # 2. Tokenize into raw integer tensor lists
        #inputs = tokenizer(prompt_text, return_tensors="pt")
        inputs = tokenizer(text=prompt_text, return_tensors="pt")
        input_ids = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)

        with torch.no_grad():
            # Added images=None to explicitly shut off the transformers image_utils scanner
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=40,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                images=None, # <--- CRITICAL FIX
            )

        # Isolate the newly generated tokens
        generated_tokens = out[0][input_ids.shape[1]:]
        raw_pred = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Clean and extract category
        pred_cat = extract_category(raw_pred)

        y_true.append(str(true_cat).strip())
        y_pred.append(pred_cat)

    # ---- 4. Calculate Metrics ----
    print("\n" + "="*40)
    print(" BASE MODEL ZERO-SHOT EVALUATION RESULTS ")
    print("="*40)

    # Get unique sorted list of all unique categories present in ground truth
    unique_labels = sorted(list(set(y_true)))

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Total Test Samples: {len(y_true)}")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Macro F1-Score:     {macro_f1:.4f}")
    print("\n--- Detailed Classification Report ---")
    # zero_division=0 prevents crashes if base model completely misses a category
    print(classification_report(y_true, y_pred, labels=unique_labels, zero_division=0))

    print("\n--- Confusion Matrix ---")
    cm = confusion_matrix(y_true, y_pred, labels=unique_labels)

    # Print clean confusion matrix table with labels
    print(f"{'True \\ Pred':<25}", end="")
    for label in unique_labels:
        # short label preview to prevent wrapping
        print(f"| {label[:8]:<8}", end="")
    print("\n" + "-" * (25 + 11 * len(unique_labels)))

    for i, label in enumerate(unique_labels):
        print(f"{label[:23]:<25}", end="")
        for j in range(len(unique_labels)):
            print(f"| {cm[i][j]:<8}", end="")
        print()

if __name__ == "__main__":
    main()

Loading base model: unsloth/Qwen3.5-0.8B


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading test data from: test.jsonl


Generating test split: 0 examples [00:00, ? examples/s]


Running evaluation on test data...


100%|██████████| 1200/1200 [29:34<00:00,  1.48s/it]


 BASE MODEL ZERO-SHOT EVALUATION RESULTS 
Total Test Samples: 1200
Accuracy:           0.2517
Macro F1-Score:     0.1455

--- Detailed Classification Report ---
                               precision    recall  f1-score   support

Arts, Crafts & Party Supplies       0.16      0.20      0.18        80
                   Automotive       0.83      0.06      0.12        80
                Baby Products       0.70      0.26      0.38        80
       Beauty & Personal Care       0.61      0.64      0.62        80
      Electronics & Computers       0.40      0.03      0.05        80
     Fashion, Shoes & Luggage       0.54      0.40      0.46        80
           Health & Household       0.18      0.81      0.29        80
               Home & Kitchen       0.17      0.46      0.25        80
      Industrial & Scientific       0.00      0.00      0.00        80
                 Pet Supplies       1.00      0.01      0.02        80
                   Smart Home       1.00      0.04      